### Link github: https://github.com/henrynlh/Artificial-Intelligence.git

In [12]:
import random

In [14]:
import random

def input_floor():
    m = int(input("Nhập số dòng (m): "))
    n = int(input("Nhập số cột (n): "))

    # Tỉ lệ xuất hiện
    obstacle_rate = 0.2  # 20% là vật cản x
    dirty_rate = 0.4    # 40% là ô bẩn 1
    # còn lại là ô sạch 0

    floor = []

    for i in range(m):
        row = []
        for j in range(n):
            r = random.random() #random số ngẫu nhiên từ 0 đến gần 1

            if r < obstacle_rate:
                row.append("x")      # vật cản
            elif r < obstacle_rate + dirty_rate:
                row.append(1)        # ô bẩn
            else:
                row.append(0)        # ô sạch

        floor.append(row)

    print(f"Ma trận {m}x{n} được tạo ngẫu nhiên:")
    print("0 = sạch, 1 = bẩn, x = vật cản")

    for row in floor:
        print(row)

    return floor, m, n

In [15]:
def get_possible_moves(pos, floor, m, n):
    x, y = pos
    moves = []

    # đi lên: hàng x - 1, cột y
    if x > 0 and floor[x - 1][y] != "x":
        moves.append("up")

    # đi xuống: hàng x + 1, cột y
    if x < m - 1 and floor[x + 1][y] != "x":
        moves.append("down")

    # đi trái: hàng x, cột y - 1
    if y > 0 and floor[x][y - 1] != "x":
        moves.append("left")

    # đi phải: hàng x, cột y + 1
    if y < n - 1 and floor[x][y + 1] != "x":
        moves.append("right")

    return moves

In [16]:
def apply_move(pos, move):
    x, y = pos
    if move == "up":
        x -= 1
    elif move == "down":
        x += 1
    elif move == "left":
        y -= 1
    elif move == "right":
        y += 1
    return (x, y)

In [17]:
def print_floor(floor):
    for row in floor:
        print(row)

In [23]:
def random_clean_prioritize_dirty(floor, m, n):
    # Chỉ random vị trí ban đầu ở ô không phải vật cản
    valid_positions = []

    for i in range(m):
        for j in range(n):
            if floor[i][j] != "x":
                valid_positions.append((i, j))

    if not valid_positions:
        print("Ma trận toàn vật cản, máy hút bụi không thể bắt đầu.")
        return

    pos = random.choice(valid_positions)
    step = 0

    print("\nTrạng thái ban đầu:")
    print_floor(floor)
    print("Vị trí ban đầu của máy hút bụi:", pos)

    x0, y0 = pos
    if floor[x0][y0] == 1:
        floor[x0][y0] = 0
        print(f"\nVị trí ban đầu bẩn ({x0},{y0}) → Đã hút bụi")
        print("Trạng thái hiện tại sau khi hút vị trí ban đầu:")
        print_floor(floor)

    # Dùng dictionary để lưu trạng thái và lần roll xuất hiện
    visited_states = {}

    # Lưu trạng thái ban đầu là roll 0
    current_state = (pos, tuple(tuple(row) for row in floor))
    visited_states[current_state] = step

    # Nếu sau khi hút vị trí ban đầu mà không còn ô bẩn
    if all(cell != 1 for row in floor for cell in row):
        print("\nTẤT CẢ CÁC Ô ĐÃ SẠCH! Hoàn thành.")
        print("\nTổng số lần roll:", step)
        return

    while True:
        step += 1
        print(f"\n========== LẦN ROLL {step} ==========")

        # Lấy các hướng có thể đi, bỏ qua vật cản x
        possible_moves = get_possible_moves(pos, floor, m, n)

        # Nếu bị kẹt, không còn hướng nào đi được
        if not possible_moves:
            print("Máy hút bụi bị kẹt, xung quanh toàn vật cản hoặc biên.")
            break

        dirty_moves = []

        for move in possible_moves:
            new_pos = apply_move(pos, move)

            if floor[new_pos[0]][new_pos[1]] == 1:
                dirty_moves.append(move)

        if dirty_moves:
            selected_move = random.choice(dirty_moves)
            print("Ưu tiên đi hướng có ô bẩn:", selected_move)
        else:
            selected_move = random.choice(possible_moves)
            print("Không có ô bẩn xung quanh, đi random:", selected_move)

        pos = apply_move(pos, selected_move)
        x, y = pos

        if floor[x][y] == 1:
            floor[x][y] = 0
            print(f"Đã hút bụi tại ô ({x},{y})")
        elif floor[x][y] == 0:
            print(f"Ô ({x},{y}) đã sạch")
        else:
            print(f"Ô ({x},{y}) là vật cản")

        print("Trạng thái hiện tại:")
        print_floor(floor)

        # Kiểm tra nếu không còn ô bẩn
        if all(cell != 1 for row in floor for cell in row):
            print("\nTẤT CẢ CÁC Ô ĐÃ SẠCH! Hoàn thành.")
            break

        # Tạo trạng thái hiện tại gồm vị trí robot + ma trận
        current_state = (pos, tuple(tuple(row) for row in floor))

        # Nếu trạng thái này đã từng xuất hiện thì báo rõ lặp ở roll nào
        if current_state in visited_states:
            previous_step = visited_states[current_state]

            print(f"\nTRẠNG THÁI BỊ LẶP!")
            print(f"Trạng thái hiện tại ở LẦN ROLL {step}.")
            print(f"Trạng thái này đã từng xuất hiện ở LẦN ROLL {previous_step}.")
            print(f"Vị trí robot bị lặp: {pos}")
            print("Dừng chương trình để tránh lặp vô hạn.")
            break

        # Nếu chưa từng xuất hiện thì lưu lại lần roll hiện tại
        visited_states[current_state] = step

    print("\nTổng số lần roll:", step)

In [24]:
floor, m, n = input_floor()
random_clean_prioritize_dirty(floor, m, n)

Ma trận 4x4 được tạo ngẫu nhiên:
0 = sạch, 1 = bẩn, x = vật cản
[0, 0, 'x', 0]
[0, 0, 1, 0]
[1, 0, 1, 0]
[0, 0, 1, 1]

Trạng thái ban đầu:
[0, 0, 'x', 0]
[0, 0, 1, 0]
[1, 0, 1, 0]
[0, 0, 1, 1]
Vị trí ban đầu của máy hút bụi: (2, 2)

Vị trí ban đầu bẩn (2,2) → Đã hút bụi
Trạng thái hiện tại sau khi hút vị trí ban đầu:
[0, 0, 'x', 0]
[0, 0, 1, 0]
[1, 0, 0, 0]
[0, 0, 1, 1]

========== LẦN ROLL 1 ==========
Ưu tiên đi hướng có ô bẩn: up
Đã hút bụi tại ô (1,2)
Trạng thái hiện tại:
[0, 0, 'x', 0]
[0, 0, 0, 0]
[1, 0, 0, 0]
[0, 0, 1, 1]

========== LẦN ROLL 2 ==========
Không có ô bẩn xung quanh, đi random: right
Ô (1,3) đã sạch
Trạng thái hiện tại:
[0, 0, 'x', 0]
[0, 0, 0, 0]
[1, 0, 0, 0]
[0, 0, 1, 1]

========== LẦN ROLL 3 ==========
Không có ô bẩn xung quanh, đi random: left
Ô (1,2) đã sạch
Trạng thái hiện tại:
[0, 0, 'x', 0]
[0, 0, 0, 0]
[1, 0, 0, 0]
[0, 0, 1, 1]

TRẠNG THÁI BỊ LẶP!
Trạng thái hiện tại ở LẦN ROLL 3.
Trạng thái này đã từng xuất hiện ở LẦN ROLL 1.
Vị trí robot bị lặp: (1, 2)